# Workshop #3 — Gold Layer Parquet Summaries Exploration (Governance & Storytelling)
**Course:** Programación para Análisis de Datos (Data Analysis Programming) — Semester 2026-II  
**Team 1:** Natural Hazard Monitoring Systems — Earthquake Activity Monitoring  
**Authors:** Juan Córdoba (20211020097), Angelo Ibáñez (20212020007), Mauricio Sánchez (20212020045)  

---

## Objective
This notebook loads and previews the two **Gold Layer Parquet datasets** generated by the PySpark pipeline (`src/gold_pyspark.py` / `gold_processing_dag.py`) in `datalake_gold/`:
1. `governance_YYYYMMDD_HHMMSS.parquet` — Data Governance, Data Quality & Structural Health KPIs.
2. `storytelling_YYYYMMDD_HHMMSS.parquet` — Functional User Storytelling Aggregations & Seismic Activity Anomaly Index (SAAI).


In [1]:
import pandas as pd
from pathlib import Path

GOLD_DIR = Path("datalake_gold") if Path("datalake_gold").exists() else Path("../datalake_gold")
gov_files = sorted(GOLD_DIR.glob("governance_*.parquet"))
story_files = sorted(GOLD_DIR.glob("storytelling_*.parquet"))

latest_gov = gov_files[-1]
latest_story = story_files[-1]

print(f"Latest Governance Gold Parquet:   {latest_gov.name} ({latest_gov.stat().st_size / 1024.0:.2f} KB)")
print(f"Latest Storytelling Gold Parquet: {latest_story.name} ({latest_story.stat().st_size / 1024.0:.2f} KB)")

gov_df = pd.read_parquet(latest_gov)
story_df = pd.read_parquet(latest_story)

print(f"\nGovernance DataFrame shape:   {gov_df.shape[0]} rows x {gov_df.shape[1]} columns")
print(f"Storytelling DataFrame shape: {story_df.shape[0]} rows x {story_df.shape[1]} columns")


Latest Governance Gold Parquet:   governance_20261121_045048.parquet (6.57 KB)
Latest Storytelling Gold Parquet: storytelling_20261121_045048.parquet (9.49 KB)

Governance DataFrame shape:   113 rows x 13 columns
Storytelling DataFrame shape: 84 rows x 14 columns


## 1. Governance Summary Parquet Inspection
Previews executive summary metrics, null/pinned-depth rates per field, volume breakdown by source, and IQR outlier rates.


In [2]:
print("=== Governance KPI Categories ===")
print(gov_df["kpi_category"].value_counts())

print("\n=== [A] Executive Governance Summary KPIs ===")
cols_gov = ["metric_name", "metric_value", "secondary_value", "threshold_value", "compliance_status", "unit"]
print(gov_df[gov_df["kpi_category"] == "summary"][cols_gov].to_string(index=False))

print("\n=== [B] Null & Default-Pinning Rates per Field ===")
print(gov_df[gov_df["kpi_category"] == "null_rate_per_field"][cols_gov].to_string(index=False))

print("\n=== [C] Outlier Rates per Numeric Field (IQR) ===")
out_cols = ["metric_name", "metric_value", "secondary_value", "min_value", "median_value", "max_value", "compliance_status"]
print(gov_df[gov_df["kpi_category"] == "outlier_rate_per_numeric_field"][out_cols].to_string(index=False))

print("\n=== [D] Volume & Place Text Length Statistics by Source ===")
txt_cols = ["source_id", "metric_name", "metric_value", "min_value", "median_value", "max_value", "unit"]
print(gov_df[gov_df["kpi_category"].isin(["volume_by_source", "text_length_statistics"])][txt_cols].to_string(index=False))


=== Governance KPI Categories ===
kpi_category
volume_over_time                  81
null_rate_per_field               13
summary                            7
volume_by_source                   4
text_length_statistics             4
outlier_rate_per_numeric_field     4
Name: count, dtype: int64

=== [A] Executive Governance Summary KPIs ===
                  metric_name  metric_value  secondary_value  threshold_value compliance_status    unit
    total_silver_records_read       4786.00             7.00            100.0              PASS records
   unique_deduplicated_events       3806.00          4786.00            100.0              PASS records
intra_lake_duplicate_rate_pct         20.48           980.00             50.0              PASS percent
   schema_compliance_rate_pct        100.00          3806.00             99.0              PASS percent
   pinned_10km_depth_rate_pct         31.69          1206.00             35.0              PASS percent
 quarantined_records_detected     

## 2. Storytelling Summary Parquet Inspection
Previews the functional user aggregations: Gutenberg-Richter severity class breakdown, top active seismic regions, multi-source catalog comparison, focal depth regimes, and daily SAAI anomaly scores.


In [3]:
print("=== Storytelling Aggregation Types ===")
print(story_df["aggregation_type"].value_counts())

s_cols = ["dimension_key", "event_count", "percentage", "mean_magnitude", "max_magnitude", "mean_depth_km", "log10_energy"]
print("\n=== [A] Earthquake Severity Tier Distribution ===")
print(story_df[story_df["aggregation_type"] == "severity_distribution"][s_cols].to_string(index=False))

print("\n=== [B] Focal Depth Regime Breakdown ===")
print(story_df[story_df["aggregation_type"] == "depth_regime_breakdown"][s_cols].to_string(index=False))

print("\n=== [C] Top 10 Most Active Seismic Regions ===")
top_cols = ["dimension_key", "event_count", "percentage", "mean_magnitude", "max_magnitude", "alert_tier"]
print(story_df[story_df["aggregation_type"] == "top_regions_keywords"][top_cols].head(10).to_string(index=False))

print("\n=== [D] Multi-Source Catalog Comparison ===")
src_cols = ["dimension_key", "secondary_dimension", "event_count", "percentage", "mean_magnitude", "max_magnitude"]
print(story_df[story_df["aggregation_type"] == "source_comparison"][src_cols].to_string(index=False))

print("\n=== [E] Peak Daily Seismic Activity Anomaly Index (SAAI) Windows ===")
saai_cols = ["period_date", "event_count", "mean_magnitude", "max_magnitude", "log10_energy", "saai_score", "alert_tier"]
print(story_df[story_df["aggregation_type"] == "temporal_trend_saai"].sort_values("saai_score", ascending=False)[saai_cols].head(10).to_string(index=False))


=== Storytelling Aggregation Types ===
aggregation_type
temporal_trend_saai       44
source_comparison         17
top_regions_keywords      15
severity_distribution      5
depth_regime_breakdown     3
Name: count, dtype: int64

=== [A] Earthquake Severity Tier Distribution ===
      dimension_key  event_count  percentage  mean_magnitude  max_magnitude  mean_depth_km  log10_energy
 Micro/Minor (<3.0)         2058       54.07           1.605           2.99          18.55         7.208
    Light (3.0-4.4)          310        8.15           3.621           4.40          48.09        10.232
 Moderate (4.5-5.4)         1347       35.39           4.776           5.40          43.80        11.964
   Strong (5.5-6.4)           84        2.21           5.774           6.40          37.19        13.461
Major/Great (>=6.5)            7        0.18           6.629           6.80          55.71        14.743

=== [B] Focal Depth Regime Breakdown ===
           dimension_key  event_count  percentage 

## 3. Raw Tabular Preview (`head(10)`) of Both Gold Parquet Datasets
Direct `pd.read_parquet()` DataFrame preview of `governance_*.parquet` and `storytelling_*.parquet`.


In [4]:
# Preview first 10 rows of Gold Governance Parquet
gov_df.head(10)


,compliance_status,computed_at,kpi_category,max_value,median_value,metric_name,metric_value,min_value,period_date,secondary_value,source_id,threshold_value,unit
0,PASS,2026-11-21T04:50:48Z,null_rate_per_field,100.0,0.0,event_id,0.0,0.0,ALL,0.0,ALL,5.0,percent
1,PASS,2026-11-21T04:50:48Z,null_rate_per_field,100.0,0.0,raw_time,0.0,0.0,ALL,0.0,ALL,5.0,percent
2,PASS,2026-11-21T04:50:48Z,null_rate_per_field,100.0,0.0,magnitude,0.0,0.0,ALL,0.0,ALL,5.0,percent
3,PASS,2026-11-21T04:50:48Z,null_rate_per_field,100.0,0.0,mag_type,0.0,0.0,ALL,0.0,ALL,5.0,percent
4,PASS,2026-11-21T04:50:48Z,null_rate_per_field,100.0,0.0,place,0.0,0.0,ALL,0.0,ALL,5.0,percent
5,PASS,2026-11-21T04:50:48Z,null_rate_per_field,100.0,0.0,longitude,0.0,0.0,ALL,0.0,ALL,5.0,percent
6,PASS,2026-11-21T04:50:48Z,null_rate_per_field,100.0,0.0,latitude,0.0,0.0,ALL,0.0,ALL,5.0,percent
7,PASS,2026-11-21T04:50:48Z,null_rate_per_field,100.0,0.0,depth_km,0.0,0.0,ALL,0.0,ALL,5.0,percent
8,PASS,2026-11-21T04:50:48Z,null_rate_per_field,100.0,0.0,tsunami,0.0,0.0,ALL,0.0,ALL,5.0,percent
9,PASS,2026-11-21T04:50:48Z,null_rate_per_field,100.0,0.0,source_file,0.0,0.0,ALL,0.0,ALL,5.0,percent


In [5]:
# Preview first 10 rows of Gold Storytelling Parquet
story_df.head(10)


,aggregation_type,alert_tier,computed_at,dimension_key,event_count,log10_energy,max_magnitude,mean_depth_km,mean_magnitude,narrative_note,percentage,period_date,saai_score,secondary_dimension
0,severity_distribution,INFO,2026-11-21T04:50:48Z,Micro/Minor (<3.0),2058,7.208,2.99,18.55,1.605,Gutenberg-Richter magnitude class distribution,54.07,ALL,0.000,ALL
1,severity_distribution,INFO,2026-11-21T04:50:48Z,Light (3.0-4.4),310,10.232,4.40,48.09,3.621,Gutenberg-Richter magnitude class distribution,8.15,ALL,0.000,ALL
2,severity_distribution,INFO,2026-11-21T04:50:48Z,Moderate (4.5-5.4),1347,11.964,5.40,43.80,4.776,Gutenberg-Richter magnitude class distribution,35.39,ALL,0.000,ALL
3,severity_distribution,INFO,2026-11-21T04:50:48Z,Strong (5.5-6.4),84,13.461,6.40,37.19,5.774,Gutenberg-Richter magnitude class distribution,2.21,ALL,0.000,ALL
4,severity_distribution,INFO,2026-11-21T04:50:48Z,Major/Great (>=6.5),7,14.743,6.80,55.71,6.629,Gutenberg-Richter magnitude class distribution,0.18,ALL,0.000,ALL
5,temporal_trend_saai,NORMAL,2026-11-21T04:50:48Z,2026-01-01,53,14.394,6.00,35.11,4.877,Daily SAAI=0.21 (NORMAL),1.39,2026-01-01,0.208,NORMAL
6,temporal_trend_saai,NORMAL,2026-11-21T04:50:48Z,2026-01-02,28,14.871,6.50,19.88,4.846,Daily SAAI=0.49 (NORMAL),0.74,2026-01-02,0.485,NORMAL
7,temporal_trend_saai,NORMAL,2026-11-21T04:50:48Z,2026-01-03,35,14.068,5.90,55.00,4.760,Daily SAAI=-0.14 (NORMAL),0.92,2026-01-03,-0.141,NORMAL
8,temporal_trend_saai,NORMAL,2026-11-21T04:50:48Z,2026-01-04,44,13.983,5.70,28.61,4.793,Daily SAAI=-0.17 (NORMAL),1.16,2026-01-04,-0.169,NORMAL
9,temporal_trend_saai,NORMAL,2026-11-21T04:50:48Z,2026-01-05,18,13.369,5.20,78.61,4.756,Daily SAAI=-0.79 (NORMAL),0.47,2026-01-05,-0.791,NORMAL
